# **셀레니움**
셀레니움(Selenium)은 파이썬 코드로 실제 웹 브라우저(Chrome, Edge 등)를 자동으로 제어할 수 있게 해주는 웹 자동화 라이브러리입니다. 일반적인 requests 크롤링은 서버에서 받은 HTML만 분석하지만, Selenium은 브라우저를 직접 실행하여 버튼 클릭, 검색 입력, 스크롤, 로그인, 페이지 이동 등의 사용자 동작을 그대로 수행할 수 있기 때문에 JavaScript로 동적으로 생성되는 데이터까지 가져올 수 있습니다. 따라서 멜론, 유튜브, 쇼핑몰처럼 JavaScript 렌더링이 많은 사이트의 크롤링이나 웹 자동화 테스트에서 매우 많이 사용됩니다. 보통 webdriver.Chrome()으로 브라우저를 실행하고, find_element()로 요소를 찾으며, click(), send_keys() 등을 통해 자동화 작업을 수행합니다.

In [15]:
import requests
from bs4 import BeautifulSoup

url = "http://127.0.0.1:5500/7.html"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
fruits = soup.select("#fruit") # 불러올 당시에는 비어있기 때문에 내용을 가져올 수 없다.
print(fruits)

[]


In [16]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

driver = webdriver.Chrome() # Chrome(): 크롬 브라우저를 자동화 시키는 객체
url = "http://127.0.0.1:5500/7.html"
driver.get(url)

# JavaScript 실행 대기
time.sleep(2)
fruits = driver.find_elements(By.CLASS_NAME, "fruit")

for fruit in fruits:
    print(fruit.text)

driver.quit()

KeyboardInterrupt: 

> 👉 find_elements()는 Selenium에서 HTML 요소를 여러 개 찾을 때 사용하는 메서드입니다. 하나의 요소만 찾는 find_element()와 달리, 조건에 맞는 요소들을 리스트 형태로 모두 반환합니다. 예를 들어 여러 개의 이미지, 게시글, 테이블 행(tr) 등을 반복해서 크롤링할 때 매우 자주 사용됩니다. 찾는 방식은 By.ID, By.CLASS_NAME, By.CSS_SELECTOR, By.XPATH 등 다양한 선택자를 사용할 수 있으며, 반환 결과는 리스트이므로 for문으로 반복 처리하는 경우가 많습니다. 또한 find_element()는 요소를 찾지 못하면 에러가 발생하지만, find_elements()는 빈 리스트([])를 반환하기 때문에 반복 크롤링에서 더 안정적으로 사용되는 경우도 많습니다.

# **2.멜론 아티스트 곡 리스트 크롤링**

### ※ xpath
XPath는 XML 또는 HTML 문서 내에서 특정 요소나 속성을 선택하기 위해 사용되는 경로 표현 언어입니다. 웹 크롤링이나 자동화 도구에서 주로 사용되며, 요소를 효율적으로 찾을 수 있도록 도와줍니다. 일반적인 XPath는 특정 위치나 속성을 기준으로 요소를 선택하는 상대적인 경로를 사용합니다. 또한 full xpath는 루트 요소에서 시작하여 대상 요소까지의 절대적인 경로를 나타냅니다. 따라서 문서 구조가 변경되면 경로가 깨질 가능성이 높습니다.

In [1]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def melon_search_from_main(keyword):
    options = Options()
    options.add_argument("--start-maximized") # 브라우저 크기에 따라 반응형에 의해 html이 다를 수 있으므로 최대화 시킨다.

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10) # 웹브라우저를 최대 10초 기다린다. 조건을 만족하면 그 전에 들어갈 수 있다.

    data = []

    try:
        driver.get("https://www.melon.com/")
        time.sleep(2)

        search_box = wait.until(
            EC.presence_of_element_located((By.ID, "top_search")) # presence_of_element_located : 특정 요소가 찾아질때까지 기다린다.
        ) # top_search는 검색창

        search_box.clear() # 지우고
        search_box.send_keys(keyword) # 쓰고
        search_box.send_keys(Keys.ENTER) # 엔터

        time.sleep(3) # 3초 기다림

        song_tab = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, '//*[@id="divCollection"]/ul/li[3]/a/span')
                
            )
        )
        song_tab.click()

        time.sleep(3)

        song_table = wait.until(
            EC.presence_of_element_located( # presence_of_element_located : 클릭할 수 있을 때까지 기다린다.
                (By.XPATH, '//*[@id="frm_defaultList"]/div/table')
            )
        )

        rows = song_table.find_elements(By.CSS_SELECTOR, "tbody tr")

        print("찾은 행 개수:", len(rows))

        for row in rows:
            try:
                cols = row.find_elements(By.TAG_NAME, "td")

                if len(cols) < 5:
                    continue

                # 곡명 td
                title_text = cols[2].text.strip()
                title_lines = [
                    line.strip()
                    for line in title_text.split("\n")
                    if line.strip()
                ]

                title = ""

                for line in title_lines:
                    if (
                        "재생" not in line
                        and "담기" not in line
                        and "상세정보" not in line
                        and not line.startswith("Title")
                    ):
                        title = line
                        break

                # Title 줄이 더 정확한 경우 보정
                for line in title_lines:
                    if line.startswith("Title "):
                        title = line.replace("Title ", "").strip()
                        break

                # 아티스트
                artist_text = cols[3].text.strip()
                artist_lines = [
                    line.strip()
                    for line in artist_text.split("\n")
                    if line.strip()
                ]
                artist = artist_lines[0] if artist_lines else ""

                # 앨범
                album_text = cols[4].text.strip()
                album_lines = [
                    line.strip()
                    for line in album_text.split("\n")
                    if line.strip()
                ]
                album = album_lines[0] if album_lines else ""

                # 좋아요 수
                like = ""
                try:
                    like = row.find_element(
                        By.CSS_SELECTOR,
                        "button.like span.cnt"
                    ).text.strip()
                except:
                    pass

                if title:
                    data.append({
                        "곡명": title,
                        "아티스트": artist,
                        "앨범": album,
                        "좋아요수": like
                    })

            except Exception as e:
                print("행 처리 오류:", e)

        df = pd.DataFrame(data)

        if not df.empty:
            df.index = df.index + 1

        file_name = f"melon_{keyword}_songs.csv"
        df.to_csv(file_name, encoding="utf-8-sig")

        print(f"CSV 저장 완료: {file_name}")
        print(f"총 {len(df)}곡 수집 완료")

        return df

    finally:
        driver.quit()

df = melon_search_from_main("신의키스")
df

찾은 행 개수: 6
CSV 저장 완료: melon_신의키스_songs.csv
총 6곡 수집 완료


,곡명,아티스트,앨범,좋아요수
1,이젠 내가 더 사랑해 (PJ1),신의키스,The Bucket List,5
2,헤어질 준비,신의키스,The Bucket List,5
3,너 없는 생일,신의키스,The Bucket List,4
4,버킷 리스트,신의키스,The Bucket List,5
5,서로를 가장 잘 아는 남,신의키스,The Bucket List,6
6,이젠 내가 더 사랑해 (PJ2),신의키스,The Bucket List,4


In [1]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def melon_search_from_main(keyword):
    options = Options()
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    data = []

    try:
        driver.get("https://www.melon.com/")
        time.sleep(2)
        search_box = wait.until(
            EC.presence_of_element_located((By.ID, "top_search"))
        )

        search_box.clear()
        search_box.send_keys(keyword)
        search_box.send_keys(Keys.ENTER)

        time.sleep(3)

        song_tab = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, '//*[@id="divCollection"]/ul/li[3]/a/span')
                
            )
        )
        song_tab.click()

        time.sleep(3)

        song_table = wait.until(
            EC.presence_of_element_located( # presence_of_element_located : 클릭할 수 있을 때까지 기다린다.
                (By.XPATH, '//*[@id="frm_defaultList"]/div/table')
            )
        )

        curpage = 1
        while True:
            
            page_tab = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, f'//*[@id="pageObjNavgation"]/div/span/a[{curpage}]')
                )
            )
            if page_tab:
                
                page_tab.click()
                time.sleep(2)
                rows = song_table.find_elements(By.CSS_SELECTOR, "tbody tr")

                print("찾은 행 개수:", len(rows))

                for row in rows:
                    try:
                        cols = row.find_elements(By.TAG_NAME, "td")

                        if len(cols) < 5:
                            continue

                        # 곡명 td
                        title_text = cols[2].text.strip()
                        title_lines = [
                            line.strip()
                            for line in title_text.split("\n")
                            if line.strip()
                        ]
                        print(title_lines)

                        title = ""

                        for line in title_lines:
                            if (
                                "재생" not in line
                                and "담기" not in line
                                and "상세정보" not in line
                                and not line.startswith("Title")
                            ):
                                title = line
                                break

                        # Title 줄이 더 정확한 경우 보정
                        for line in title_lines:
                            if line.startswith("Title "):
                                title = line.replace("Title ", "").strip()
                                break

                        # 아티스트
                        artist_text = cols[3].text.strip()
                        artist_lines = [
                            line.strip()
                            for line in artist_text.split("\n")
                            if line.strip()
                        ]
                        artist = artist_lines[0] if artist_lines else ""

                        # 앨범
                        album_text = cols[4].text.strip()
                        album_lines = [
                            line.strip()
                            for line in album_text.split("\n")
                            if line.strip()
                        ]
                        album = album_lines[0] if album_lines else ""

                        # 좋아요 수
                        like = ""
                        try:
                            like = row.find_element(
                                By.CSS_SELECTOR,
                                "button.like span.cnt"
                            ).text.strip()
                        except:
                            pass

                        if title:
                            data.append({
                                "곡명": title,
                                "아티스트": artist,
                                "앨범": album,
                                "좋아요수": like
                            })

                    except Exception as e:
                        print("행 처리 오류:", e)
                curpage += 1
                time.sleep(2)
                
            else:
                
                break

        df = pd.DataFrame(data)

        if not df.empty:
            df.index = df.index + 1

        return df

    finally:
        driver.quit()

df = melon_search_from_main("악뮤")
df


StaleElementReferenceException: Message: stale element reference: stale element not found in the current frame
  (Session info: chrome=149.0.7827.199); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff78ad03fa5+14925]
	chromedriver!GetHandleVerifier [0x7ff78ad04000+14980]
	chromedriver!(No symbol) [0x7ff78a84793d]
	chromedriver!(No symbol) [0x7ff78a84f8b1]
	chromedriver!(No symbol) [0x7ff78a852875]
	chromedriver!(No symbol) [0x7ff78a85293f]
	chromedriver!(No symbol) [0x7ff78a8a122a]
	chromedriver!(No symbol) [0x7ff78a8a1dac]
	chromedriver!(No symbol) [0x7ff78a895e2c]
	chromedriver!(No symbol) [0x7ff78a8957c6]
	chromedriver!(No symbol) [0x7ff78a8ef39b]
	chromedriver!(No symbol) [0x7ff78a89401c]
	chromedriver!(No symbol) [0x7ff78a894f43]
	chromedriver!GetHandleVerifier [0x7ff78b2e7591+5f7f11]
	chromedriver!GetHandleVerifier [0x7ff78b2e1902

In [ ]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException


def extract_current_page(driver, wait):
    data = []

    try:
        song_table = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, '//*[@id="frm_defaultList"]/div/table')
            )
        )
    except TimeoutException:
        return []

    rows = song_table.find_elements(By.CSS_SELECTOR, "tbody tr")

    for row in rows:
        cols = row.find_elements(By.TAG_NAME, "td")

        if len(cols) < 5:
            continue

        title_lines = [
            line.strip()
            for line in cols[2].text.split("\n")
            if line.strip()
        ]

        title = ""

        for line in title_lines:
            if line.startswith("Title "):
                title = line.replace("Title ", "").strip()
                break

        if not title:
            for line in title_lines:
                if (
                    "재생" not in line
                    and "담기" not in line
                    and "상세정보" not in line
                    and not line.startswith("Title")
                ):
                    title = line
                    break

        artist_lines = [
            line.strip()
            for line in cols[3].text.split("\n")
            if line.strip()
        ]
        artist = artist_lines[0] if artist_lines else ""

        album_lines = [
            line.strip()
            for line in cols[4].text.split("\n")
            if line.strip()
        ]
        album = album_lines[0] if album_lines else ""

        try:
            like = row.find_element(
                By.CSS_SELECTOR,
                "button.like span.cnt"
            ).text.strip()
        except:
            like = ""

        if title:
            data.append({
                "곡명": title,
                "아티스트": artist,
                "앨범": album,
                "좋아요수": like
            })

    return data


def melon_search_all_pages(keyword, max_page=30):
    options = Options()
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    all_data = []

    try:
        driver.get("https://www.melon.com/")
        time.sleep(2)

        search_box = wait.until(
            EC.presence_of_element_located((By.ID, "top_search"))
        )

        search_box.clear()
        search_box.send_keys(keyword)
        search_box.send_keys(Keys.ENTER)

        time.sleep(3)

        song_tab = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, '//*[@id="divCollection"]/ul/li[3]/a/span')
            )
        )

        song_tab.click()
        time.sleep(3)

        for page in range(1, max_page + 1):
            page_data = extract_current_page(driver, wait)

            if not page_data:
                print(f"{page}페이지 데이터가 없어 종료합니다.")
                break

            all_data.extend(page_data)
            print(f"{page}페이지 크롤링 완료: {len(page_data)}곡")

            next_start_index = page * 50 + 1

            try:
                driver.execute_script( # execute_script 자바스크립트 실행할 수 있게 해준다.
                    f"pageObj.sendPage('{next_start_index}');" # scrollto를 이용해 스크롤 조작도 가능
                )
                time.sleep(3)

            except Exception as e:
                print("다음 페이지 이동 실패:", e)
                break

        df = pd.DataFrame(all_data)

        if not df.empty:
            df = df.drop_duplicates(
                subset=["곡명", "아티스트", "앨범"]
            )
            df.index = df.index + 1

        file_name = f"melon_{keyword}_all_songs.csv"

        df.to_csv(
            file_name,
            encoding="utf-8-sig"
        )

        print(f"CSV 저장 완료: {file_name}")
        print(f"총 {len(df)}곡 수집 완료")

        return df

    finally:
        driver.quit()

melon_search_all_pages("악뮤", 30)

1페이지 크롤링 완료: 50곡
2페이지 크롤링 완료: 50곡
3페이지 크롤링 완료: 8곡
4페이지 데이터가 없어 종료합니다.
CSV 저장 완료: melon_악뮤_all_songs.csv
총 108곡 수집 완료


,곡명,아티스트,앨범,좋아요수
1,"기쁨, 슬픔, 아름다운 마음",AKMU (악뮤),개화,"85,804"
2,소문의 낙원,AKMU (악뮤),개화,"83,830"
3,DINOSAUR,AKMU (악뮤),SUMMER EPISODE,"205,786"
4,"어떻게 이별까지 사랑하겠어, 널 사랑하는 거지",AKMU (악뮤),항해,"495,833"
5,오랜 날 오랜 밤,AKMU (악뮤),사춘기 하 (思春記 下),"246,653"
...,...,...,...,...
104,Crescendo (크레셴도),AKMU (악뮤),2013 MelOn Music Awards Nominees,936
105,콩떡빙수 (Radio Edit),AKMU (악뮤),콩떡빙수,595
106,뜨거운 안녕,AKMU (악뮤),SBS K팝 스타 시즌2 TOP 2,814
107,외국인의 고백,AKMU (악뮤),SBS K팝 스타 시즌2 TOP 3 Part.1,"1,096"


# **3. 스타벅스 서울 전체 매장 크롤링**

In [47]:
import time
import re
import pandas as pd

from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup

def fetch_starbucks():
    url = "https://www.starbucks.co.kr/index.do"
    driver = webdriver.Chrome()
    driver.maximize_window()
    driver.get(url)
    time.sleep(2)

    # 메뉴 이동
    action = ActionChains(driver)

    first_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#gnb > div > nav > div > ul > li.gnb_nav03" # Store 메뉴
    )

    second_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#gnb > div > nav > div > ul > li.gnb_nav03 > div > div > div > ul:nth-child(1) > li:nth-child(3) > a"
    )

    # 마우스 이동
    action.move_to_element(first_tag).move_to_element(second_tag).click().perform()

    # 서울 선택
    seoul_tag = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((
            By.CSS_SELECTOR,
            "#container > div > form > fieldset > div > section > article.find_store_cont > article > article:nth-child(4) > div.loca_step1 > div.loca_step1_cont > ul > li:nth-child(1) > a"
        ))
    )

    seoul_tag.click()

    # 구 목록 로딩 대기
    WebDriverWait(driver, 5).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "set_gugun_cd_btn")
        )
    )

    gu_elements = driver.find_elements( # 전체를 가져와 모든 구가 리스트로 담긴다.
        By.CLASS_NAME,
        "set_gugun_cd_btn"
    )

    # 전체 선택
    gu_elements[0].click() # 리스트의 첫번째를 클릭

    # 매장 목록 로딩 대기
    WebDriverWait(driver, 5).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "quickResultLstCon")
        )
    )

    # HTML 가져오기
    req = driver.page_source # 불러온 상황의 html을 가져온다.

    soup = BeautifulSoup(req, "html.parser")

    stores = soup.find(
        'ul',
        'quickSearchResultBoxSidoGugun'
    ).find_all('li')

    # 데이터 저장 리스트
    store_list = []
    addr_list = []
    lat_list = []
    lng_list = []

    # 데이터 추출
    for store in stores:
        store_name = store.find("strong").text
        store_addr = store.find("p").text
        # 전화번호 제거
        store_addr = re.sub(
            r'\d{4}-\d{4}$',
            '',
            store_addr
        ).strip()

        store_lat = store['data-lat']
        store_lng = store['data-long']

        store_list.append(store_name)
        addr_list.append(store_addr)
        lat_list.append(store_lat)
        lng_list.append(store_lng)

    # 데이터프레임 생성
    df = pd.DataFrame({
        'store': store_list,
        'addr': addr_list,
        'lat': lat_list,
        'lng': lng_list
    })

    driver.quit()

    return df

# 함수 실행
starbucks_df = fetch_starbucks()

# CSV 저장
starbucks_df.to_csv(
    "starbucks_seoul.csv",
    index=False,
    encoding='utf-8-sig'
)

print("데이터가 starbucks_seoul.csv 파일로 저장되었습니다.")
print(starbucks_df.head())

데이터가 starbucks_seoul.csv 파일로 저장되었습니다.
       store                        addr         lat          lng
0  역삼아레나빌딩       서울특별시 강남구 언주로 425 (역삼동)   37.501087   127.043069
1   논현역사거리      서울특별시 강남구 강남대로 538 (논현동)   37.510178   127.022223
2  신사역성일빌딩      서울특별시 강남구 강남대로 584 (논현동)  37.5139309  127.0206057
3   국기원사거리      서울특별시 강남구 테헤란로 125 (역삼동)   37.499517   127.031495
4   대치재경빌딩    서울특별시 강남구 남부순환로 2947 (대치동)   37.494668   127.062583


In [41]:
fetch_starbucks()